# FLAG 2027 — NB-4: ReDimNet2 voice encoders (round 3)

One question, one variable: **does multilingual pretraining of the voice encoder help the Bangla cells?**
Two ReDimNet2-B6 checkpoints that differ ONLY in training data (MIT licence, github.com/PalabraAI/redimnet2):

| short | checkpoint | training data |
|---|---|---|
| `rdn6vox` | b6 / lm / `vox2` | VoxCeleb2 (mostly English) |
| `rdn6multi` | b6 / lm / `vb2+vox2+cnc2_v0` | VoxBlink2 + VoxCeleb2 + CN-Celeb2 (multilingual) |

Everything else is the EXP-007 recipe, and the crops reuse NB-1's `crop_plan_train.npy`, so `*_mix` runs are
directly comparable with `r2_ecapa192_mix`.

**Settings:** GPU T4, **Internet ON** (torch.hub download). **Inputs:** the raw dataset (wav), `flag2027-features`,
`flag2027-feats-v2` (crop plan). ~45-60 min total.

⚠️ ReDimNet2 has no length mask: zero padding changes its embedding (local test: cos 0.63 with 3 s of padding),
so extraction runs **one file at a time** (no batching). Slower, but exact.

Output: `feats_v3/` (new arrays + manifest), `submissions/r3_*.zip`, `*_scores.npz`, `*_matrix.npz`
(full face x voice score matrices for graph refinement), `runs.csv`, `p0_unimodal_v3.csv`.

In [1]:
import os, sys
from pathlib import Path
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| working", WORKDIR)

KAGGLE | input /kaggle/input | working /kaggle/working


In [2]:
# Writes the helper modules next to this notebook (no .py upload needed).
import base64
open('flag_lib.py', 'wb').write(base64.b64decode(''.join('''
...
cmV0dXJuIHBhdGgK
'''.split())))
open('flag_extract.py', 'wb').write(base64.b64decode(''.join('''
...
YWQoZmMsIG1tYXBfbW9kZT0iciIpLnNoYXBlKQo=
'''.split())))
open('flag_v2.py', 'wb').write(base64.b64decode(''.join('''
...
IGNoZWNrcG9pbnQsIGRvY3MvREFUQS5tZCkiKQo=
'''.split())))
print('written:', ['flag_lib.py', 'flag_extract.py', 'flag_v2.py'])

written: ['flag_lib.py', 'flag_extract.py', 'flag_v2.py']


## 1. Extract ReDimNet2-B6 (vox2) and (vb2+vox2+cnc2), one file at a time

In [3]:
import numpy as np, pandas as pd, torch, time, json
import flag_extract as X
OUT3 = WORKDIR / "feats_v3"; OUT3.mkdir(exist_ok=True)
root = X.media_root(IN, WORKDIR / "raw")
S = X.index_splits(root)
plan = sorted(IN.rglob("crop_plan_train.npy"))
assert plan, "crop_plan_train.npy not found: attach flag2027-feats-v2"
# multilingual first: it is the question of this round, and it is the slow one (extra GroupNorm layers)
ENCS = [("b6", "vb2+vox2+cnc2_v0", "rdn6multi"), ("b6", "vox2", "rdn6vox")]
BUDGET_MIN = 0.5 if LOCAL else 150          # per encoder; the guard drops crops, then the encoder
DONE_ENC = []
for mn, ds, short in ENCS:
    t0 = time.time()
    model = X.load_redimnet2(mn, ds)
    try:
        X.extract_hub_voice(S, OUT3, short, model, crop_plan_file=plan[0], max_minutes=BUDGET_MIN,
                            probe_files=10 if LOCAL else 50)
        DONE_ENC.append(short)
    except X.EncoderTooSlow as e:
        print("SKIPPED:", e)
    del model; torch.cuda.empty_cache()
    print(f"{short}: {time.time() - t0:.0f}s")
print("extracted:", DONE_ENC)
X.write_manifest(OUT3, dict(encoders=ENCS, extracted=DONE_ENC, crop_plan=str(plan[0])))

23:05:38 train                 6485 rows | dur median 5.2s max 60.0s
23:05:43 no_gender/English     1008 rows | dur median 5.8s max 81.6s
23:05:48 no_gender/Bangla      1406 rows | dur median 4.7s max 43.9s
23:05:53 gender/English         982 rows | dur median 6.1s max 81.6s
23:05:59 gender/Bangla         1468 rows | dur median 4.7s max 43.9s
Downloading: "https://github.com/PalabraAI/redimnet2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/PalabraAI/redimnet2/releases/download/v1.0.0/b6-vb2+vox2+cnc2_v0-lm.pt" to /root/.cache/torch/hub/checkpoints/b6-vb2+vox2+cnc2_v0-lm.pt


100%|██████████| 48.8M/48.8M [00:00<00:00, 166MB/s]


23:06:12   rdn6multi train 0/6485 1s
23:06:16   rdn6multi speed probe: 0.087s per 5s of audio -> ETA full 23 min, crops 23 min (budget 150)
23:09:09   rdn6multi train 2000/6485 178s
23:12:02   rdn6multi train 4000/6485 352s
23:14:57   rdn6multi train 6000/6485 526s
23:15:40 train saved (6485, 192)
23:15:40   rdn6multi no_gender/English 0/1008 0s
23:17:16 no_gender/English saved (1008, 192)
23:17:16   rdn6multi no_gender/Bangla 0/1406 0s
23:18:53 no_gender/Bangla saved (1406, 192)
23:18:53   rdn6multi gender/English 0/982 0s
23:20:29 gender/English saved (982, 192)
23:20:30   rdn6multi gender/Bangla 0/1468 0s
23:22:12 gender/Bangla saved (1468, 192)
23:27:27   rdn6multi crop0 done 316s
23:32:43   rdn6multi crop1 done 315s
23:38:01   rdn6multi crop2 done 318s
23:38:01 crops saved (3, 6485, 192)
rdn6multi: 1915s


Using cache found in /root/.cache/torch/hub/PalabraAI_redimnet2_main


Downloading: "https://github.com/PalabraAI/redimnet2/releases/download/v1.0.0/b6-vox2-lm.pt" to /root/.cache/torch/hub/checkpoints/b6-vox2-lm.pt


100%|██████████| 48.5M/48.5M [00:00<00:00, 183MB/s]


23:38:02   rdn6vox train 0/6485 0s
23:38:06   rdn6vox speed probe: 0.057s per 5s of audio -> ETA full 15 min, crops 15 min (budget 150)
23:40:49   rdn6vox train 2000/6485 167s
23:43:34   rdn6vox train 4000/6485 332s
23:46:19   rdn6vox train 6000/6485 497s
23:47:00 train saved (6485, 192)
23:47:00   rdn6vox no_gender/English 0/1008 0s
23:48:31 no_gender/English saved (1008, 192)
23:48:31   rdn6vox no_gender/Bangla 0/1406 0s
23:50:03 no_gender/Bangla saved (1406, 192)
23:50:03   rdn6vox gender/English 0/982 0s
23:51:36 gender/English saved (982, 192)
23:51:36   rdn6vox gender/Bangla 0/1468 0s
23:53:13 gender/Bangla saved (1468, 192)
23:58:27   rdn6vox crop0 done 314s
00:03:41   rdn6vox crop1 done 314s
00:08:57   rdn6vox crop2 done 316s
00:08:57 crops saved (3, 6485, 192)
rdn6vox: 1856s
extracted: ['rdn6multi', 'rdn6vox']
00:08:57 manifest: 12 arrays


{'created': '2026-09-26 00:08',
 'files': {'voice_rdn6multi_gender_Bangla.npy': [1468, 192],
  'voice_rdn6multi_gender_English.npy': [982, 192],
  'voice_rdn6multi_no_gender_Bangla.npy': [1406, 192],
  'voice_rdn6multi_no_gender_English.npy': [1008, 192],
  'voice_rdn6multi_train.npy': [6485, 192],
  'voice_rdn6multicrop_train.npy': [3, 6485, 192],
  'voice_rdn6vox_gender_Bangla.npy': [1468, 192],
  'voice_rdn6vox_gender_English.npy': [982, 192],
  'voice_rdn6vox_no_gender_Bangla.npy': [1406, 192],
  'voice_rdn6vox_no_gender_English.npy': [1008, 192],
  'voice_rdn6vox_train.npy': [6485, 192],
  'voice_rdn6voxcrop_train.npy': [3, 6485, 192]},
 'encoders': [('b6', 'vb2+vox2+cnc2_v0', 'rdn6multi'),
  ('b6', 'vox2', 'rdn6vox')],
 'extracted': ['rdn6multi', 'rdn6vox'],
 'crop_plan': '/kaggle/input/datasets/chnnhnnguyn/flag2027-feats-v2/feats/crop_plan_train.npy'}

## 2. Unimodal sanity + EXP-007 recipe on each encoder (full / Bangla-length crops)

In [4]:
os.environ["FLAG_DATA"] = str(IN)
os.environ["FLAG_FEATS_EXTRA"] = str(OUT3)
import flag_v2 as V
V.polarity_preflight()
store = V.FeatureStore(IN)
# unimodal sanity (speaker EER on train, no training): a broken extraction shows up here immediately
rows = []
for n in ["given", "ecapa192", "rdn6vox", "rdn6multi"]:
    if store.has("voice", n):
        rows.append(dict(feat=n, **V.unimodal_eer(store.voice(n, "train"), store.spk, store.gmap)))
        C = store.voice_crops(n)
        if C is not None:
            rows.append(dict(feat=n + " (Bangla-length crop)", **V.unimodal_eer(C[0], store.spk, store.gmap)))
P0 = pd.DataFrame(rows); P0.to_csv(WORKDIR / "p0_unimodal_v3.csv", index=False); print(P0.to_string(index=False))

REC = dict(V.BASE, head="mlp", drop=0.3, emb=128, epochs=40)      # = EXP-007 recipe, same as r2_*
RUNS = {f"r3_{v}{'_mix' if m else ''}": dict(REC, voice=v, voice_view="mix" if m else "full")
        for v in ["rdn6multi", "rdn6vox"] if store.has("voice", v)
        for m in (False, True) if not m or store.voice_crops(v) is not None}
print("runs:", list(RUNS))
N_MODELS = 2 if LOCAL else 10
if LOCAL:
    RUNS = {k: dict(v, epochs=2) for k, v in RUNS.items()}
SUBS = WORKDIR / "submissions"; SUBS.mkdir(exist_ok=True)
for name, cfg in RUNS.items():
    out = SUBS / f"{name}.zip"
    if out.exists():
        print("skip (exists)", out.name); continue
    t0 = time.time()
    sc = V.dev_scores(store, cfg, n_models=N_MODELS, cca_w=0.25, matrix_path=SUBS / f"{name}_matrix.npz")
    np.savez(SUBS / f"{name}_scores.npz", **{"/".join(k): v for k, v in sc.items()})
    V.write_zip(store, out, sc); V.check_zip(store, out)
    pd.DataFrame([dict(zip=out.name, sec=round(time.time() - t0), cfg=json.dumps(cfg))]).to_csv(
        SUBS / "runs.csv", mode="a", header=not (SUBS / "runs.csv").exists(), index=False)
    print(f"{name}: {time.time() - t0:.0f}s")
print(sorted(p.name for p in SUBS.glob("*.zip")))

00:08:58 polarity: internal scores are higher = same; write_submission writes -score (LOWER = same, measured on CodaBench: -d2 57.83 vs +d2 42.17 with one checkpoint, docs/DATA.md)
00:10:17 feature dir: /kaggle/input/datasets/chnnhnnguyn/flag2027-feats-v2/feats | 33 arrays
00:10:17 feature dir: /kaggle/working/feats_v3 | 12 arrays
                          feat   ng    g
                         given 4.77 5.93
                      ecapa192 4.33 5.33
 ecapa192 (Bangla-length crop) 4.60 5.87
                       rdn6vox 3.57 4.17
  rdn6vox (Bangla-length crop) 3.97 4.33
                     rdn6multi 3.17 3.50
rdn6multi (Bangla-length crop) 3.23 3.70
runs: ['r3_rdn6multi', 'r3_rdn6multi_mix', 'r3_rdn6vox', 'r3_rdn6vox_mix']
00:14:39 zip OK: /kaggle/working/submissions/r3_rdn6multi.zip
r3_rdn6multi: 260s
00:18:13 zip OK: /kaggle/working/submissions/r3_rdn6multi_mix.zip
r3_rdn6multi_mix: 214s
00:21:46 zip OK: /kaggle/working/submissions/r3_rdn6vox.zip
r3_rdn6vox: 213s
00:25:19 zip OK: 

In [5]:
import shutil
shutil.rmtree(WORKDIR / "raw", ignore_errors=True)      # never ship extracted media in the output
tot = sum(p.stat().st_size for p in WORKDIR.rglob("*") if p.is_file()) / 1e9
print(f"output {tot:.2f} GB. Download: submissions/ (zips + npz) and feats_v3/ (for the next rounds).")

output 0.09 GB. Download: submissions/ (zips + npz) and feats_v3/ (for the next rounds).
